# Step 13 — Recover laterality

**Settings: Accelerator = `None` (CPU), Internet = `Off`.** Attach the competition data.

## What we're fixing

Only **52.5%** of studies carry a `Laterality` DICOM tag. The other 47.5% get mirrored to the right-knee convention purely on geometric inference — and if that inference is wrong, medial and lateral swap for those studies.

The evidence that it *is* wrong:

| | medial | lateral | gap |
|---|---|---|---|
| Meniscus | 0.766 | 0.668 | 0.098 |
| Osteoarthritis | 0.837 | 0.745 | 0.092 |

Two independent findings, near-identical deficit on the lateral side. A model that can't tell the compartments apart falls back on the base rate, and medial pathology is far more common (2,505 vs 1,612 positives) — so medial holds up while lateral collapses.

## The idea

DICOM `ImagePositionPatient` is in **LPS** coordinates: +x is the patient's **left**, +y posterior, +z superior. A knee sits lateral to the midline, so the sign of x should reveal which leg it is.

**We have 210 studies with a ground-truth tag to check this against before trusting it.** If the rule agrees with the tag, apply it to the other half. If it doesn't, fall back to a learned classifier.

This notebook only *decides the rule*. It writes a laterality table; the cache rebuild uses it.

### 1 · Harvest geometry and tags

In [ ]:
import os, glob, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, pydicom
from multiprocessing import Pool

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train = pd.read_csv(f"{ROOT}/train.csv")
studies = train["StudyInstanceUID"].tolist()
print(f"studies: {len(studies):,}")

def probe(suid):
    """Return per-study laterality tag (if any) plus geometric summaries."""
    d = f"{ROOT}/train_series/{suid}"
    if not os.path.isdir(d):
        return dict(StudyInstanceUID=suid)
    tag, xs, xr, planes = "", [], [], []
    for s in sorted(os.listdir(d)):
        p = os.path.join(d, s)
        fs = sorted(os.listdir(p))
        if not fs: continue
        for f in (fs[0], fs[len(fs)//2], fs[-1]):
            try:
                ds = pydicom.dcmread(os.path.join(p, f), stop_before_pixels=True)
            except Exception:
                continue
            l = str(getattr(ds, "Laterality", "") or "").strip().upper()
            if l and not tag: tag = l[0]
            ipp = getattr(ds, "ImagePositionPatient", None)
            iop = getattr(ds, "ImageOrientationPatient", None)
            if ipp is not None:
                xs.append(float(ipp[0]))
            if ipp is not None and iop is not None:
                r, c = np.array(iop[:3], float), np.array(iop[3:], float)
                n = np.cross(r, c)
                # x of the series centre, offset along the slice normal
                xr.append(float(ipp[0]) + 0.0)
                planes.append(int(np.argmax(np.abs(n))))
    return dict(StudyInstanceUID=suid, tag=tag,
                x_mean=float(np.mean(xs)) if xs else np.nan,
                x_min=float(np.min(xs)) if xs else np.nan,
                x_max=float(np.max(xs)) if xs else np.nan,
                n_obs=len(xs))

t0 = time.time()
with Pool(4) as pool:
    rows = []
    for i, r in enumerate(pool.imap(probe, studies, chunksize=16), 1):
        rows.append(r)
        if i % 500 == 0:
            print(f"  {i}/{len(studies)}  {(time.time()-t0)/i:.3f}s/study  "
                  f"eta {(len(studies)-i)*(time.time()-t0)/i/60:.0f} min", flush=True)
g = pd.DataFrame(rows)
print(f"\ndone in {(time.time()-t0)/60:.1f} min")
print(f"tag present: {(g.tag.fillna('') != '').mean():.1%}")
print(g.tag.value_counts(dropna=False).to_string())

### 2 · Does the x-coordinate predict the side?

Validated only on studies that carry a tag. `x_mean` is the average x of the sampled slices; if the LPS convention holds, left knees should sit at positive x and right knees at negative.

In [ ]:
lab = g[(g.tag.isin(["L","R"])) & g.x_mean.notna()].copy()
print(f"studies with tag AND geometry: {len(lab):,}")

print("\nx_mean distribution by tag:")
print(lab.groupby("tag")["x_mean"].describe()[["count","mean","std","25%","50%","75%"]].round(1).to_string())

from sklearn.metrics import roc_auc_score, accuracy_score
y = (lab.tag == "L").astype(int).values
auc = roc_auc_score(y, lab.x_mean.values)
print(f"\nAUC of x_mean predicting LEFT: {auc:.3f}")
print("  (0.5 = useless, >0.9 = strong, <0.5 = sign is inverted)")

best = None
for thr in np.percentile(lab.x_mean, np.arange(2, 99, 1)):
    for sign in (1, -1):
        pred = ((lab.x_mean.values - thr) * sign > 0).astype(int)
        a = accuracy_score(y, pred)
        if best is None or a > best[0]:
            best = (a, thr, sign)
acc, thr, sign = best
print(f"\nbest rule: LEFT if (x_mean - {thr:.1f}) * {sign} > 0   -> accuracy {acc:.1%}")

### 3 · Verdict and fallback

Above ~90% accuracy the rule is good enough to use. Between 75 and 90 it helps but leaves residual error. Below 75 the geometry has been destroyed by de-identification and we need a different approach.

The fallback, if geometry fails: train a small classifier on the 52.5% with tags — predicting left vs right from a knee image is an easy task, and half your data is labelled.

In [ ]:
if auc < 0.5:
    print("!! AUC below 0.5 -- the sign convention is inverted, flip it")
q = max(auc, 1 - auc)
print(f"discriminative power (max(auc, 1-auc)): {q:.3f}   accuracy {acc:.1%}\n")

if acc >= 0.90:
    print("VERDICT: geometry works. Apply the rule to the 47.5% missing a tag.")
elif acc >= 0.75:
    print("VERDICT: geometry partially works. Usable, but a learned classifier would be better.")
else:
    print("VERDICT: geometry is not recoverable this way.")
    print("Options: (a) train a small L/R classifier on the tagged half,")
    print("         (b) drop mirroring entirely and feed laterality as a feature,")
    print("         (c) keep mirroring only where the tag exists, leave the rest untouched.")

### 4 · Write the laterality table

One row per study: the tag if present, otherwise the geometric prediction, with a column recording which source was used so training can weight or inspect them separately.

In [ ]:
g["lat_final"] = ""
g["lat_src"]   = ""

has_tag = g.tag.isin(["L","R"])
g.loc[has_tag, "lat_final"] = g.loc[has_tag, "tag"]
g.loc[has_tag, "lat_src"]   = "tag"

geo_ok = (~has_tag) & g.x_mean.notna()
pred_left = ((g.loc[geo_ok, "x_mean"] - thr) * sign > 0)
g.loc[geo_ok, "lat_final"] = np.where(pred_left, "L", "R")
g.loc[geo_ok, "lat_src"]   = "geometry"

g.loc[g.lat_final == "", "lat_src"] = "unknown"

out = g[["StudyInstanceUID","lat_final","lat_src","tag","x_mean","n_obs"]]
out.to_csv("/kaggle/working/laterality.csv", index=False)

print(out.lat_src.value_counts().to_string())
print()
print(out[out.lat_final != ""].lat_final.value_counts().to_string())
print(f"\nresolved: {(out.lat_final != '').mean():.1%} of studies")
print("\nwrote laterality.csv")